In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim


In [24]:
#  set random seed for reporducibility
torch.manual_seed(42)

In [25]:
df = pd.read_csv('pytorch/assets/fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [26]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [27]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [28]:
X_train = X_train/255.0
X_test = X_test/255.0

In [ ]:
# create custom dataset class
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


In [ ]:
train_dataset = CustomDataset(X_train, y_train)


In [ ]:
test_dataset = CustomDataset(X_test, y_test)


In [32]:
# loader
train_loader =  DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [34]:
# NN class
class MyNN(nn.Module):
    def __init__(self, num_features):

        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p = 0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout( p = 0.3),
            nn.Linear(64, 10),
            # by default softmax involved hai
        )
    def forward(self, x):

        return self.model(x)   


In [35]:
epochs = 100
learning_rate = 0.1

In [37]:
model = MyNN(X_train.shape[1])

# loss function
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay = 1e-4)

In [ ]:
# training loop
for epoch in range(epochs):

    total_epochs_loss = 0
    for batch_features, batch_labels in train_loader:

        # forward pass
        outputs = model(batch_features)
        # calc loss
        loss = criterion(outputs, batch_labels)
        # back prop
        optimizer.zero_grad()
        loss.backward()
        # update params
        optimizer.step()

        total_epochs_loss = total_epochs_loss + loss.item()

    average_loss = total_epochs_loss / len(train_loader)
    print(f'Epoch: {epoch + 1}, Loss: {average_loss}')


In [39]:
# evaluation
model.eval()


MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [40]:
# eval code
total = 0
correct = 0
with torch.no_grad():

    for batch_features, batch_labels in test_loader:

        outputs = model(batch_features)
        # we get 32,10 for 32 images batch and probability for 10 classifications. extract labels with highest prob
        _, predicted = torch.max(outputs, 1)
        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)        


0.8266666666666667
